# PASO 0 — ETL: ingesta, preprocesamiento y muestreo (BugsRepo)

**Examen Parcial 3 · Proyecto de Investigación 2 · UNI**

Este notebook implementa el **primer paso numerado** del entregable (`0_…`), alineado con la sesión
**«Ingesta y Preprocesamiento Reproducibles»** (`clases/2Curso/Maestria2_IA02.pptx`).

| Fase ETL | Concepto UNI (IA02) | Qué hace este notebook |
|----------|---------------------|------------------------|
| **E — Extract** | Ingesta desde `raw` | Carga CSV local o HuggingFace → `0_datos_crudos_…` |
| **T — Transform** | Preprocesamiento + limpieza | Normalización de columnas + embudo en 5 fases |
| **L — Load** | Datos `processed` listos | Persiste corpus, muestras, splits e informes JSON |

**Semilla reproducible:** `seed = 42` (muestreo estratificado + hold-out 70/30).

---

## Artefactos generados (misma carpeta ETL)

| # | Archivo | Rol |
|---|---------|-----|
| 0 | `0_datos_crudos_bugzilla_eclipse.csv` | Extract — copia/ingesta del dataset crudo |
| 1 | `1_embudo_limpieza_por_fases.csv` | Transform — embudo fase a fase |
| 2 | `2_corpus_limpio_bugsrepo.csv` | Load — corpus limpio (~88 008 registros) |
| 3 | `3_informe_muestreo_y_distribucion.json` | Load — estadísticas globales del pipeline |
| 4 | `4_muestra_laboratorio_n400.csv` | Load — muestra lab n=400 |
| 5 | `5_conjunto_entrenamiento_holdout_70.csv` | Load — 70 % train sobre muestra lab |
| 6 | `6_conjunto_prueba_holdout_30.csv` | Load — 30 % test sobre muestra lab |
| 7 | `7_informe_division_train_test.json` | Load — ratios, conteos y refs. slides |
| 8 | `8_conjunto_evaluacion_experimento_n400.csv` | Load — evaluación LLM sobre **muestra lab completa** (n=400) |


In [1]:
# ── Configuración reproducible ──────────────────────────────────────────
# Entrada: ninguna (solo librerías).
# Salida: variables globales de rutas y constantes usadas en todo el pipeline.
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

SEED = 42
np.random.seed(SEED)

# Mapeo Bugzilla → criticidad operacional (play / pausa / stop)
SEVERITY_MAP = {
    "blocker": "stop",
    "critical": "stop",
    "major": "pausa",
    "normal": "pausa",
    "minor": "play",
    "trivial": "play",
}
SEVERITY_EXCLUIR = {"enhancement", "n/a", "nan", "--", "unspecified", ""}
RESOLUTION_KEEP = {"FIXED", "VERIFIED", "RESOLVED"}

# Directorio ETL: ejecutar el notebook desde esta carpeta (Kernel → cwd = ETL)
ETL_DIR = Path.cwd().resolve()
if not (ETL_DIR / "0_datos_crudos_bugzilla_eclipse.csv").exists():
    _candidato = Path("0_etl_ingesta_preprocesamiento")
    if (_candidato / "0_datos_crudos_bugzilla_eclipse.csv").exists():
        ETL_DIR = _candidato.resolve()

# Nombres de salida numerados (español descriptivo)
F_RAW = ETL_DIR / "0_datos_crudos_bugzilla_eclipse.csv"
F_FUNNEL = ETL_DIR / "1_embudo_limpieza_por_fases.csv"
F_CLEAN = ETL_DIR / "2_corpus_limpio_bugsrepo.csv"
F_SAMPLE_REPORT = ETL_DIR / "3_informe_muestreo_y_distribucion.json"
F_LAB = ETL_DIR / "4_muestra_laboratorio_n400.csv"
F_TRAIN = ETL_DIR / "5_conjunto_entrenamiento_holdout_70.csv"
F_TEST = ETL_DIR / "6_conjunto_prueba_holdout_30.csv"
F_SPLIT_REPORT = ETL_DIR / "7_informe_division_train_test.json"
F_EVAL = ETL_DIR / "8_conjunto_evaluacion_experimento_n400.csv"

print(f"Directorio ETL : {ETL_DIR}")
print(f"CSV crudo      : {F_RAW.name} (existe={F_RAW.exists()})")

Directorio ETL : /home/jlpy/Documents/4Semestre/Proyecto de Investigacion 2/Tesis/tesis2/parcial3/entregable/0_etl_ingesta_preprocesamiento
CSV crudo      : 0_datos_crudos_bugzilla_eclipse.csv (existe=True)


## E — Extract (Ingesta)

> **IA02 · slide 3:** `data/raw` almacena datos originales; la ingesta debe ser reproducible y documentada.

### Fuente y licencia

| Aspecto | Detalle |
|---------|---------|
| **Dataset** | Bugzilla Eclipse Bug Reports (BugsRepo) |
| **Repositorio HF** | [`AliArshad/Bugzilla_Eclipse_Bug_Reports_Dataset`](https://huggingface.co/datasets/AliArshad/Bugzilla_Eclipse_Bug_Reports_Dataset) |
| **Fallback** | [`AliArshad/Bug_Reports_with_Sentiments`](https://huggingface.co/datasets/AliArshad/Bug_Reports_with_Sentiments) |
| **Referencia** | Acharya & Ginde (2025) — BugsRepo |
| **Licencia** | Dataset público HuggingFace — ver tarjeta del repositorio |

### Ética / PII

- Textos de bugs pueden contener rutas técnicas; **no** exportamos IDs de reportero.
- Clave interna `text_hash` (SHA-256) para trazabilidad sin duplicar texto en splits.
- Datos históricos públicos Bugzilla; no hay recolección primaria.

**Salida de esta sección:** DataFrame estandarizado + CSV crudo en `0_datos_crudos_bugzilla_eclipse.csv` si se descarga desde HF.

In [2]:
# ── E — Extract: carga local primero, HuggingFace si falta ───────────────
# Entrada: F_RAW (CSV local) o descarga HF.
# Salida: df_raw (columnas text/severity/resolution/product), data_source, n_raw.
HF_PRIMARY = "AliArshad/Bugzilla_Eclipse_Bug_Reports_Dataset"
HF_FALLBACK = "AliArshad/Bug_Reports_with_Sentiments"

BUGSREPO_COLS_VARIANTS = [
    {"text": "Short Description", "severity": "Severity Label", "resolution": "Resolution Status", "product": "Project"},
    {"text": "short_desc", "severity": "severity", "resolution": "resolution", "product": "product"},
    {"text": "text", "severity": "severity", "resolution": "resolution", "product": "product"},
]


def _normalize_columns(df: pd.DataFrame, mapping: dict[str, str]) -> pd.DataFrame:
    out = pd.DataFrame()
    for std, src in mapping.items():
        if src in df.columns:
            out[std] = df[src]
        elif std in df.columns:
            out[std] = df[std]
    return out


def _has_bugzilla_severity(series: pd.Series) -> bool:
    vals = set(series.astype(str).str.lower().str.strip().unique())
    return bool(vals & set(SEVERITY_MAP.keys()))


def load_from_huggingface(dataset_name: str) -> tuple[pd.DataFrame, str]:
    from datasets import load_dataset

    ds = load_dataset(dataset_name, split="train")
    df = ds.to_pandas()
    for mapping in BUGSREPO_COLS_VARIANTS:
        if mapping["text"] in df.columns and mapping["severity"] in df.columns:
            return _normalize_columns(df, mapping), f"HuggingFace:{dataset_name}"
    raise ValueError(f"No se pudo mapear columnas en {dataset_name}: {list(df.columns)}")


def ensure_raw_csv(path: Path) -> tuple[pd.DataFrame, str]:
    if path.is_file():
        df = pd.read_csv(path)
        for mapping in BUGSREPO_COLS_VARIANTS:
            if mapping["text"] in df.columns or "text" in df.columns:
                if "text" not in df.columns:
                    std = _normalize_columns(df, mapping)
                else:
                    std = df.copy()
                    for c in ("severity", "resolution", "product"):
                        if c not in std.columns:
                            std[c] = np.nan
                return std, f"local:{path.name}"

    print("CSV local no encontrado — descargando desde HuggingFace…")
    try:
        df, label = load_from_huggingface(HF_PRIMARY)
        if not _has_bugzilla_severity(df["severity"]):
            raise ValueError("Severities no Bugzilla en dataset primario")
    except Exception as exc:
        print(f"  Primario falló ({exc}); usando fallback…")
        df, label = load_from_huggingface(HF_FALLBACK)

    df.to_csv(path, index=False)
    print(f"  Guardado Extract → {path.name}")
    return df, label


df_raw, data_source = ensure_raw_csv(F_RAW)
n_raw = len(df_raw)
print(f"Fuente de ingesta: {data_source}")
print(f"Registros crudos (n_raw): {n_raw:,}")

Fuente de ingesta: local:0_datos_crudos_bugzilla_eclipse.csv
Registros crudos (n_raw): 88,682


### Esquema tras la ingesta

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `text` | str | Descripción corta del bug (campo NLP) |
| `severity` | str | Severidad Bugzilla (blocker…trivial) |
| `resolution` | str | Estado de resolución (FIXED, VERIFIED, …) |
| `product` | str | Proyecto Eclipse asociado |

Columnas **generadas** en Transform:

| Columna | Descripción |
|---------|-------------|
| `text_hash` | SHA-256 del texto normalizado (dedup + anti-leakage) |
| `thesis_class` | Clase operacional UNI: **play / pausa / stop** |

In [3]:
# ── Exploración rápida post-ingesta ─────────────────────────────────────
# Entrada: df_raw. Salida: vista previa en pantalla (no escribe archivos).
display(df_raw.head(3))
print("\nTipos y nulos:")
display(df_raw.dtypes.to_frame("dtype").join(df_raw.isna().sum().rename("nulos")))
print("\nSeverities crudas (top 10):")
display(df_raw["severity"].astype(str).str.lower().str.strip().value_counts().head(10))

,text,severity,resolution,product
0,Clean up user selection SQL,normal,FIXED,Bugzilla
1,Typo in error message,trivial,FIXED,Bugzilla
2,If logged out $user->can_bless gives: Not an A...,normal,FIXED,Bugzilla



Tipos y nulos:


,dtype,nulos
text,str,0
severity,str,0
resolution,str,0
product,str,0



Severities crudas (top 10):


severity
normal      72170
critical     5936
major        4573
minor        3125
trivial      2080
blocker       798
Name: count, dtype: int64

## T — Transform (Preprocesamiento y limpieza)

> **IA02 · slide 3:** `data/interim` opcional para datos intermedios; aquí registramos el embudo en CSV.

Pipeline inline (sin depender de `src/`) en **5 fases** con embudo documentado:

| Fase | Acción | Criterio |
|------|--------|----------|
| **FASE 0** | Quitar nulos | Sin `severity`/`text` vacíos; texto > 3 caracteres |
| **FASE 1** | Filtro resolución | Solo `FIXED`, `VERIFIED`, `RESOLVED` (si aplica) |
| **FASE 2** | Excluir enhancement | Severities no-bug (`enhancement`, `n/a`, …) |
| **FASE 3** | Dedup SHA-256 | Texto normalizado idéntico → un registro |
| **FASE 4** | Mapeo criticidad | Bugzilla → `thesis_class` play/pausa/stop |

**Salidas:** `df_clean`, `1_embudo_limpieza_por_fases.csv`

In [4]:
# ── T — Transform: pipeline de limpieza en 5 fases ───────────────────────
# Entrada: df_raw. Salida: df_clean, funnel_rows, n_raw (reconfirmado).


def sha256_text(text: str) -> str:
    normalized = re.sub(r"\s+", " ", str(text).lower().strip())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def registrar_funnel(
    funnel: list[dict[str, Any]],
    nombre: str,
    df_in: pd.DataFrame,
    df_out: pd.DataFrame,
    detalle: str = "",
) -> None:
    eliminados = len(df_in) - len(df_out)
    pct = len(df_out) / len(df_in) * 100 if len(df_in) > 0 else 0.0
    funnel.append(
        {
            "Fase": nombre,
            "Antes": len(df_in),
            "Después": len(df_out),
            "Eliminados": eliminados,
            "% Retenido": round(pct, 1),
            "Detalle": detalle,
        }
    )


def run_cleaning_pipeline(df_raw: pd.DataFrame) -> tuple[pd.DataFrame, list[dict[str, Any]], int]:
    funnel: list[dict[str, Any]] = []
    n_raw_local = len(df_raw)

    df = df_raw.copy()
    df["severity"] = df["severity"].astype(str).str.lower().str.strip()
    if "resolution" in df.columns:
        df["resolution"] = df["resolution"].astype(str).str.upper().str.strip()
    else:
        df["resolution"] = "FIXED"

    # FASE 0 — registros incompletos
    df_f0 = df[
        df["severity"].notna()
        & df["severity"].ne("nan")
        & df["severity"].ne("")
        & df["text"].notna()
        & (df["text"].astype(str).str.len() > 3)
    ].copy()
    registrar_funnel(funnel, "FASE 0 — Eliminar registros sin severity/texto", df, df_f0, "Campos vacíos o nulos")
    df = df_f0

    # FASE 1 — resolución válida
    if "resolution" in df.columns and df["resolution"].nunique() > 1:
        df_f1 = df[df["resolution"].isin(RESOLUTION_KEEP)].copy()
        detalle = "Solo resolution IN [FIXED, VERIFIED, RESOLVED]"
    else:
        df_f1 = df.copy()
        detalle = "Resolution no aplica o ya curado"
    registrar_funnel(funnel, "FASE 1 — Filtro Resolution=FIXED", df, df_f1, detalle)
    df = df_f1

    # FASE 2 — excluir enhancement
    df_f2 = df[~df["severity"].isin(SEVERITY_EXCLUIR)].copy()
    registrar_funnel(funnel, "FASE 2 — Exclusión de Enhancement", df, df_f2, str(SEVERITY_EXCLUIR))
    df = df_f2

    # FASE 3 — deduplicación por hash
    df["text_hash"] = df["text"].apply(sha256_text)
    df_f3 = df.drop_duplicates(subset="text_hash").copy()
    registrar_funnel(funnel, "FASE 3 — Deduplicación SHA-256", df, df_f3, "Texto normalizado idéntico")
    df = df_f3

    # FASE 4 — mapeo a thesis_class
    df["thesis_class"] = df["severity"].map(SEVERITY_MAP)
    df_f4 = df[df["thesis_class"].notna()].copy()
    unmapped = df.loc[df["thesis_class"].isna(), "severity"].unique().tolist()
    registrar_funnel(
        funnel,
        "FASE 4 — Mapeo Bugzilla → Play/Pausa/Stop",
        df,
        df_f4,
        f"Sin mapeo: {unmapped}" if unmapped else "ninguna",
    )
    df = df_f4

    return df, funnel, n_raw_local


df_clean, funnel_rows, n_raw = run_cleaning_pipeline(df_raw)
n_clean = len(df_clean)
funnel_df = pd.DataFrame(funnel_rows)

print(f"n_raw={n_raw:,} → n_clean={n_clean:,} (retención global {n_clean/n_raw*100:.1f}%)")
display(funnel_df)

n_raw=88,682 → n_clean=88,008 (retención global 99.2%)


,Fase,Antes,Después,Eliminados,% Retenido,Detalle
0,FASE 0 — Eliminar registros sin severity/texto,88682,88667,15,100.0,Campos vacíos o nulos
1,FASE 1 — Filtro Resolution=FIXED,88667,88667,0,100.0,Resolution no aplica o ya curado
2,FASE 2 — Exclusión de Enhancement,88667,88667,0,100.0,"{'', 'nan', 'enhancement', '--', 'n/a', 'unspe..."
3,FASE 3 — Deduplicación SHA-256,88667,88008,659,99.3,Texto normalizado idéntico
4,FASE 4 — Mapeo Bugzilla → Play/Pausa/Stop,88008,88008,0,100.0,ninguna


### Mapeo de criticidad (Bugzilla → `thesis_class`)

En la **FASE 4** del embudo, cada registro recibe la columna `thesis_class` ∈ {`play`, `pausa`, `stop`} mediante el diccionario `SEVERITY_MAP` definido al inicio del notebook. La tabla resumida es:

| Severidad Bugzilla | `thesis_class` | Interpretación operacional (framework UNI) |
|--------------------|----------------|--------------------------------------------|
| blocker, critical | **stop** | HITL — supervisión humana inmediata (equivalente a modo STOP / validación paso a paso) |
| major, normal | **pausa** | HOTL — abstención selectiva y revisión en checkpoints (Chow, 1970) |
| minor, trivial | **play** | HOOTL — autonomía plena del agente cuando el riesgo inferido es bajo |

---

### Justificación metodológica del mapeo severidad → criticidad

#### 1. Por qué se transforma *severity* Bugzilla en play/pausa/stop

El framework de la tesis define la **criticidad operacional** como una política de supervisión humana proporcional al riesgo (Impacto × Probabilidad × Detectabilidad, inspirada en FMEA/ISO 14971 y NASA-STD-8739.8), materializada en tres modos: **play** (HOOTL), **pausa** (HOTL) y **stop** (HITL). Para entrenar y evaluar el pipeline ETL → clasificador → experimento LLM se requiere una **etiqueta de referencia** (`thesis_class`) en cada registro.

BugsRepo/Eclipse expone la taxonomía nativa de Bugzilla (`blocker`, `critical`, `major`, `normal`, `minor`, `trivial`), que es la única anotación estructurada disponible a escala en el corpus público (Acharya & Ginde, 2025). No existe en el dataset un campo de *criticidad operacional* del framework; por ello se aplica un **mapeo determinista** de seis niveles Bugzilla a tres clases del experimento.

#### 2. En qué base se apoya el mapeo

| Fundamento | Rol en esta decisión |
|------------|----------------------|
| **Convención Bugzilla** | *Severity* clasifica la gravedad percibida del defecto en el triaje del proyecto (prioridad de resolución), no el riesgo FMEA del framework. |
| **FMEA / modos HOOTL–HOTL–HITL** (TESIS, cap. marco teórico) | La tesis exige tres bandas de supervisión; el mapeo colapsa la escala ordinal Bugzilla en tres niveles compatibles con la política play/pausa/stop. |
| **Literatura BugsRepo** (Acharya & Ginde, 2025; arXiv:2504.18806) | Usa metadatos Bugzilla —incluida *severity*— como variable supervisada en tareas de clasificación de bugs; el mapeo a tres clases es un **proxy label** aceptado en benchmarks de severidad textual. |
| **Convención del repositorio** (`tesis2/src/bugsrepo_loader.py`, `SEVERITY_MAP`) | Mapeo canónico reproducible, idéntico en loader y notebook, para garantizar trazabilidad entre ETL y experimentos. |
| **IA02 (Ingesta reproducible)** | Documentar la transformación de etiquetas es parte del preprocesamiento; las *gold labels* del conjunto `casos_gold_criticidad_v2.jsonl` (n=300, anotación humana) son el benchmark primario de criticidad real; BugsRepo es **validación externa por proxy**. |

#### 3. Observación crítica (honesta)

> **La severidad Bugzilla es una etiqueta de clasificación de resolución/prioridad del reporte**, asignada en el flujo de triaje del proyecto Eclipse, **no** una medida de **riesgo operacional** del framework (Impacto × Probabilidad × Detectabilidad).

Consecuencias metodológicas:

- Un bug *normal* puede describir un fallo de seguridad en el texto; un *blocker* puede referirse a un bloqueo de build sin impacto en producción.
- La distribución real del corpus es muy desbalanceada (`normal` ≈ 81 %); la muestra de laboratorio n=400 se estratifica **artificialmente** por `thesis_class`, no refleja prevalencia operacional.
- Métricas altas en BugsRepo **no implican** validez ecológica de la política play/pausa/stop en despliegue empresarial.

#### 4. Por qué aún lo adoptamos en el experimento

A pesar del *gap* conceptual, el mapeo es **metodológicamente defendible** como benchmark de laboratorio porque:

1. **Reproducibilidad:** corpus público, pipeline documentado, semilla fija (`seed=42`), mismo `SEVERITY_MAP` en código y notebook.
2. **Corpus de referencia en la literatura:** BugsRepo/Bugzilla Eclipse es citado y reutilizado en investigación reciente de severidad de bugs.
3. **Triaje histórico como señal débil:** equipos humanos ya ordenaron reportes en una escala ordinal; *blocker*/*critical* suelen correlacionar con interrupción de flujo de trabajo, aunque no con FMEA.
4. **Proxy label explícito:** al declarar la limitación, el experimento evalúa capacidad del pipeline para **recuperar una taxonomía externa de severidad**, no para certificar criticidad operacional real.

#### 5. Por qué puede servir como demostración del pipeline

- **Correlación semántica parcial:** descripciones de *blocker*/*critical* frecuentemente mencionan caídas, pérdida de datos o bloqueo total — señal útil para entrenar/evaluar un clasificador texto → stop.
- **Contraste con gold humano:** el conjunto `casos_gold_criticidad_v2` (300 casos, 100/clase, español) valida la política real; BugsRepo prueba **generalización a dominio distinto** (inglés, bugs Eclipse).
- **Limitaciones explícitas:** reportar F1-macro, desbalance y falsos negativos en *stop* documenta dónde el proxy falla — resultado científico válido aunque las métricas sean modestas.

| Severity Bugzilla | `thesis_class` | Razonamiento del mapeo | Limitación principal |
|-----------------|----------------|------------------------|----------------------|
| **blocker** | stop | Máxima prioridad Bugzilla; bloqueo de release/build → analogía con HITL obligatorio | No distingue bloqueo técnico de riesgo operacional en producción |
| **critical** | stop | Fallo grave en funcionalidad core; alineación semántica con escalamiento humano | Puede etiquetarse por impacto en milestone, no por Impacto×Prob×Det |
| **major** | pausa | Defecto importante pero no bloqueante; revisión antes de merge razonable | Muchos *major* son cosméticos; texto puede no reflejar HOTL |
| **normal** | pausa | Clase modal del corpus; defecto estándar en cola de resolución | Domina el 86 % del corpus; mezcla severidades heterogéneas |
| **minor** | play | Impacto acotado; correcciones de bajo riesgo | Texto breve; posible confusión con *trivial* |
| **trivial** | play | Typo, UI menor, sin efecto funcional | Triaje laxo; no garantiza autonomía HOOTL segura |

#### Por qué es aceptable como benchmark de laboratorio

El experimento no pretende sustituir la anotación humana de criticidad (gold v2) ni certificar despliegue en producción. Su función es **cerrar el ciclo ETL reproducible** sobre un corpus público masivo, generar splits estratificados (n=400, hold-out 70/30) y alimentar la evaluación comparativa baseline / VAR-1 / VAR-2 con una referencia **externa, estable y documentada**. En la literatura de predicción de severidad de bugs, usar *severity* Bugzilla como etiqueta supervisada es práctica establecida; aquí se explicita que actúa como **proxy label**, no como ground truth de riesgo FMEA.

#### Qué NO demostramos vs. qué SÍ demostramos

| Qué **NO** demostramos con este mapeo | Qué **SÍ** demostramos |
|---------------------------------------|-------------------------|
| Equivalencia entre severidad Bugzilla y criticidad operacional FMEA del framework | Que el pipeline ETL ingiere, limpia, mapea y persiste un corpus listo para modelado con trazabilidad fase a fase |
| Validez ecológica de play/pausa/stop en un entorno empresarial real | Que un clasificador puede aprender señales textuales correlacionadas con la taxonomía Bugzilla (benchmark externo) |
| Superioridad de VAR-1/VAR-2 sobre anotación humana gold | Reproducibilidad del flujo Extract → Transform → Load alineado con IA02 y con el informe parcial 3 |
| Calibración de umbrales FMEA a partir de metadatos Eclipse | Honestidad metodológica: limitaciones del proxy declaradas antes de interpretar métricas |

In [5]:
# ── Tabla de mapeo (documentación + conteos en corpus limpio) ───────────
# Entrada: df_clean, SEVERITY_MAP (sin modificar la lógica del pipeline).
# Salida: tablas en pantalla — no escribe archivos adicionales.

# Razonamiento y limitación por severidad (solo documentación; no altera FASE 4)
JUSTIFICACION_MAPEO = [
    {
        "severity_bugzilla": "blocker",
        "thesis_class": "stop",
        "razonamiento": "Máxima prioridad Bugzilla; bloqueo de release/build → analogía HITL",
        "limitacion": "No distingue bloqueo técnico de riesgo operacional en producción",
    },
    {
        "severity_bugzilla": "critical",
        "thesis_class": "stop",
        "razonamiento": "Fallo grave en funcionalidad core; escalamiento humano razonable",
        "limitacion": "Prioridad de milestone, no Impacto×Probabilidad×Detectabilidad FMEA",
    },
    {
        "severity_bugzilla": "major",
        "thesis_class": "pausa",
        "razonamiento": "Defecto importante no bloqueante; revisión en checkpoint (HOTL)",
        "limitacion": "Muchos major son cosméticos; texto puede no justificar pausa",
    },
    {
        "severity_bugzilla": "normal",
        "thesis_class": "pausa",
        "razonamiento": "Defecto estándar en cola de triaje; supervisión intermedia",
        "limitacion": "Clase modal (~81 % crudo); mezcla heterogénea de gravedad real",
    },
    {
        "severity_bugzilla": "minor",
        "thesis_class": "play",
        "razonamiento": "Impacto acotado; correcciones de bajo riesgo (HOOTL)",
        "limitacion": "Descripciones breves; confusión posible con trivial",
    },
    {
        "severity_bugzilla": "trivial",
        "thesis_class": "play",
        "razonamiento": "Typo/UI menor sin efecto funcional",
        "limitacion": "Triaje laxo; no garantiza autonomía HOOTL segura",
    },
]

justificacion_df = pd.DataFrame(JUSTIFICACION_MAPEO)
justificacion_df["n_en_corpus"] = justificacion_df["severity_bugzilla"].map(
    lambda s: int((df_clean["severity"] == s).sum())
)

print("Tabla metodológica — Severity Bugzilla → thesis_class (FASE 4):")
display(justificacion_df)

# Verificación: el mapeo documentado coincide con SEVERITY_MAP del pipeline
assert set(zip(justificacion_df["severity_bugzilla"], justificacion_df["thesis_class"])) == set(
    SEVERITY_MAP.items()
), "JUSTIFICACION_MAPEO debe ser coherente con SEVERITY_MAP"

mapeo_df = pd.DataFrame(
    [
        {"severity_bugzilla": k, "thesis_class": v, "n_en_corpus": int((df_clean["severity"] == k).sum())}
        for k, v in SEVERITY_MAP.items()
    ]
)
print("\nConteos por severidad (resumen):")
display(mapeo_df)

dist = df_clean["thesis_class"].value_counts().sort_index()
dist_pct = (dist / n_clean * 100).round(2)
dist_table = pd.DataFrame({"conteo": dist, "porcentaje": dist_pct})
print("\nDistribución final por thesis_class:")
display(dist_table)

Tabla metodológica — Severity Bugzilla → thesis_class (FASE 4):


,severity_bugzilla,thesis_class,razonamiento,limitacion,n_en_corpus
0,blocker,stop,Máxima prioridad Bugzilla; bloqueo de release/...,No distingue bloqueo técnico de riesgo operaci...,793
1,critical,stop,Fallo grave en funcionalidad core; escalamient...,"Prioridad de milestone, no Impacto×Probabilida...",5846
2,major,pausa,Defecto importante no bloqueante; revisión en ...,Muchos major son cosméticos; texto puede no ju...,4536
3,normal,pausa,Defecto estándar en cola de triaje; supervisió...,Clase modal (~81 % crudo); mezcla heterogénea ...,71654
4,minor,play,Impacto acotado; correcciones de bajo riesgo (...,Descripciones breves; confusión posible con tr...,3118
5,trivial,play,Typo/UI menor sin efecto funcional,Triaje laxo; no garantiza autonomía HOOTL segura,2061



Conteos por severidad (resumen):


,severity_bugzilla,thesis_class,n_en_corpus
0,blocker,stop,793
1,critical,stop,5846
2,major,pausa,4536
3,normal,pausa,71654
4,minor,play,3118
5,trivial,play,2061



Distribución final por thesis_class:


,conteo,porcentaje
thesis_class,,
pausa,76190,86.57
play,5179,5.88
stop,6639,7.54


## T — Transform (Muestreo y división hold-out)

Referencias UNI:

| Tema | Slide | Decisión aquí |
|------|-------|---------------|
| Validación estratificada | IA04 · slide 12 | Muestreo y split por `thesis_class` |
| Hold-out vs CV | IA04 · slide 23 | Hold-out 70/30 sobre muestra lab |
| Train/test | IA04 · slide 6 | Transformaciones ajustadas solo en train (PASO 2) |
| Ratio 70/30 | IA04 (1Curso) · slide 11 | 70 % entrenamiento / 30 % prueba |
| Tamaño muestra lab | Cochran (referencia) | n₀ ≈ 384 → **n=400** por redondeo práctico |
| Evaluación LLM | Experimento parcial 3 | **n=400** — muestra lab completa (sin subconjunto artificial) |

$$n_0 = \frac{Z^2 \cdot p \cdot (1-p)}{E^2} \approx 384$$

Se adopta **n=400** (entero redondo, cercano a Cochran) en lugar de 384 para evitar que el
tamaño parezca copiado literal de la fórmula.

El hold-out 70/30 se aplica sobre la **muestra lab** (n=400), no sobre los ~88 k del corpus.

**Por qué n=400 para el experimento LLM:** el tamaño se inspira en Cochran (~384) y es
estadísticamente aceptable para inferencia sobre la población limpia (~88 k). Se descarta un
subconjunto piloto n=40 del test porque reduce artificialmente el poder estadístico y facilita
críticas metodológicas. El archivo `8_conjunto_evaluacion_experimento_n400.csv` replica la
muestra lab completa; el split train (~280) / test (~120) se conserva para RAG e historial,
pero la comparación baseline / var1 / var2 se ejecuta sobre los **400** registros.


In [6]:
# ── T — Muestreo estratificado n=400 + hold-out 70/30 + conjunto evaluación ──
# Entrada: df_clean. Salida: muestra_lab, split_train, split_test, conjunto_eval, split_report.
# LAB_N=400: redondeo práctico de Cochran (~384); muestreo estratificado desde corpus limpio.
from sklearn.model_selection import train_test_split

CLASSES = ("play", "pausa", "stop")
TRAIN_RATIO = 0.70
TEST_RATIO = 0.30
LAB_N = 400  # ≈ Cochran n₀≈384; entero redondo elegido por conveniencia metodológica

Z = 1.96
P = 0.50
E = 0.05
N_POP = len(df_clean)
n_cochran_inf = (Z**2 * P * (1 - P)) / (E**2)
n_cochran_fin = n_cochran_inf / (1 + (n_cochran_inf - 1) / N_POP)

print(f"Cochran (Z={Z}, p={P}, E={E}): n₀={n_cochran_inf:.1f} (referencia)")
print(f"Corrección finita (N={N_POP:,}): n={n_cochran_fin:.1f}")
print(f"Muestra lab adoptada: LAB_N={LAB_N} (≈ Cochran, redondeo práctico)")


def stratified_sample(df: pd.DataFrame, n_total: int, seed: int) -> pd.DataFrame:
    props = df["thesis_class"].value_counts(normalize=True).reindex(CLASSES, fill_value=0)
    raw = props * n_total
    counts = raw.round().astype(int)
    diff = n_total - counts.sum()
    if diff != 0:
        remainders = (raw - raw.astype(int)).sort_values(ascending=(diff < 0))
        for cls in remainders.index:
            if diff == 0:
                break
            counts[cls] += 1 if diff > 0 else -1
            diff += -1 if diff > 0 else 1
    parts = []
    for cls in CLASSES:
        pool = df[df["thesis_class"] == cls]
        n_want = int(counts[cls])
        if len(pool) < n_want:
            raise ValueError(f"Pool insuficiente para {cls}: {len(pool)} < {n_want}")
        parts.append(pool.sample(n=n_want, random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)


muestra_lab = stratified_sample(df_clean, LAB_N, SEED)
print(f"\nMuestra lab: n={len(muestra_lab)}")
display(muestra_lab["thesis_class"].value_counts().sort_index())

split_train, split_test = train_test_split(
    muestra_lab,
    test_size=TEST_RATIO,
    random_state=SEED,
    stratify=muestra_lab["thesis_class"],
)
print(f"\nSplit hold-out: train={len(split_train)} ({TRAIN_RATIO:.0%}) · test={len(split_test)} ({TEST_RATIO:.0%})")
display(pd.DataFrame({
    "train": split_train["thesis_class"].value_counts().sort_index(),
    "test": split_test["thesis_class"].value_counts().sort_index(),
}))

# Evaluación LLM = muestra lab completa (n=400). Train reservado para RAG/history.
conjunto_eval = muestra_lab.copy()
eval_note = (
    f"Experimento LLM sobre muestra lab completa (n={len(conjunto_eval)}). "
    f"Hold-out train={len(split_train)} / test={len(split_test)} se mantiene para RAG; "
    f"evaluación baseline/var1/var2 usa los {len(conjunto_eval)} registros."
)

print(f"\nConjunto evaluación experimento LLM: n={len(conjunto_eval)}")
print(eval_note)
display(conjunto_eval["thesis_class"].value_counts().sort_index())

split_report = {
    "seed": SEED,
    "cochran": {"Z": Z, "p": P, "E": E, "n0_infinite": round(n_cochran_inf, 2), "n_finite": round(n_cochran_fin, 2)},
    "lab_n_adopted": LAB_N,
    "lab_n_note": "n=400 ≈ Cochran n₀≈384; entero redondo por redondeo práctico",
    "n_lab": int(len(muestra_lab)),
    "train_ratio": TRAIN_RATIO,
    "test_ratio": TEST_RATIO,
    "n_train": int(len(split_train)),
    "n_test": int(len(split_test)),
    "train_distribution": split_train["thesis_class"].value_counts().sort_index().to_dict(),
    "test_distribution": split_test["thesis_class"].value_counts().sort_index().to_dict(),
    "eval_experimento_n": int(len(conjunto_eval)),
    "eval_experimento_distribution": conjunto_eval["thesis_class"].value_counts().sort_index().to_dict(),
    "eval_experimento_source": F_LAB.name,
    "eval_experimento_note": eval_note,
    "slide_refs": [
        "2Curso/Maestria2_IA02.pptx (ingesta y preprocesamiento reproducibles)",
        "2Curso/Maestria2_IA04.pptx slide 12 (validación estratificada)",
        "2Curso/Maestria2_IA04.pptx slide 23 (holdout/CV)",
        "1Curso/Maestria_IA04.pptx slide 11 (ejemplo 70% train / 30% test)",
    ],
    "split_applied_on": F_LAB.name,
}


Cochran (Z=1.96, p=0.5, E=0.05): n₀=384.2 (referencia)
Corrección finita (N=88,008): n=382.5
Muestra lab adoptada: LAB_N=400 (≈ Cochran, redondeo práctico)

Muestra lab: n=400


thesis_class
pausa    346
play      24
stop      30
Name: count, dtype: int64


Split hold-out: train=280 (70%) · test=120 (30%)


,train,test
thesis_class,,
pausa,242,104
play,17,7
stop,21,9



Conjunto evaluación experimento LLM: n=400
Experimento LLM sobre muestra lab completa (n=400). Hold-out train=280 / test=120 se mantiene para RAG; evaluación baseline/var1/var2 usa los 400 registros.


thesis_class
pausa    346
play      24
stop      30
Name: count, dtype: int64

## L — Load (Persistencia de artefactos)

> **IA02 · slide 3:** `data/processed` contiene datos listos para modelado.

Guardamos todos los outputs numerados en la carpeta ETL. El PASO 1 del pipeline consumirá
`5_conjunto_entrenamiento_holdout_70.csv` (RAG/history), `6_conjunto_prueba_holdout_30.csv`
(hold-out de referencia) y `8_conjunto_evaluacion_experimento_n400.csv` (evaluación LLM n=400).

In [7]:
# ── L — Load: escribir CSV e informes JSON ───────────────────────────────
# Entrada: df_clean, funnel_df, muestras, split_report.
# Salida: archivos 1_… a 8_… en ETL_DIR.

funnel_df.to_csv(F_FUNNEL, index=False)
df_clean.to_csv(F_CLEAN, index=False)
muestra_lab.to_csv(F_LAB, index=False)
split_train.to_csv(F_TRAIN, index=False)
split_test.to_csv(F_TEST, index=False)
conjunto_eval.to_csv(F_EVAL, index=False)

F_SPLIT_REPORT.write_text(
    json.dumps(split_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

sample_report = {
    "seed": SEED,
    "data_source": data_source,
    "n_raw": int(n_raw),
    "n_clean": int(n_clean),
    "retention_pct": round(n_clean / n_raw * 100, 2),
    "funnel": funnel_rows,
    "class_distribution": dist.to_dict(),
    "class_distribution_pct": dist_pct.round(2).to_dict(),
    "severity_map": SEVERITY_MAP,
    "cochran": split_report["cochran"],
    "n_lab": split_report["n_lab"],
    "lab_distribution": muestra_lab["thesis_class"].value_counts().sort_index().to_dict(),
    "split": {
        "train_ratio": TRAIN_RATIO,
        "test_ratio": TEST_RATIO,
        "n_train": split_report["n_train"],
        "n_test": split_report["n_test"],
        "train_distribution": split_report["train_distribution"],
        "test_distribution": split_report["test_distribution"],
        "applied_on": split_report["split_applied_on"],
    },
    "eval_experimento_n": split_report["eval_experimento_n"],
    "eval_experimento_distribution": split_report["eval_experimento_distribution"],
    "eval_experimento_source": split_report["eval_experimento_source"],
    "eval_experimento_note": split_report["eval_experimento_note"],
    "slide_refs": split_report["slide_refs"],
    "outputs": [
        F_RAW.name,
        F_FUNNEL.name,
        F_CLEAN.name,
        F_SAMPLE_REPORT.name,
        F_LAB.name,
        F_TRAIN.name,
        F_TEST.name,
        F_SPLIT_REPORT.name,
        F_EVAL.name,
    ],
}
F_SAMPLE_REPORT.write_text(
    json.dumps(sample_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Artefactos guardados (Load):")
for name in sample_report["outputs"]:
    p = ETL_DIR / name
    size_mb = p.stat().st_size / (1024 * 1024) if p.exists() else 0
    print(f"  ✓ {name} ({size_mb:.2f} MB)")

display(Markdown(
    f"**Resumen ETL:** {n_raw:,} → **{n_clean:,}** limpios · "
    f"lab n={LAB_N} · train={len(split_train)} · test={len(split_test)} · eval LLM={len(conjunto_eval)}"
))

Artefactos guardados (Load):
  ✓ 0_datos_crudos_bugzilla_eclipse.csv (6.77 MB)
  ✓ 1_embudo_limpieza_por_fases.csv (0.00 MB)
  ✓ 2_corpus_limpio_bugsrepo.csv (12.67 MB)
  ✓ 3_informe_muestreo_y_distribucion.json (0.00 MB)
  ✓ 4_muestra_laboratorio_n400.csv (0.06 MB)
  ✓ 5_conjunto_entrenamiento_holdout_70.csv (0.04 MB)
  ✓ 6_conjunto_prueba_holdout_30.csv (0.02 MB)
  ✓ 7_informe_division_train_test.json (0.00 MB)
  ✓ 8_conjunto_evaluacion_experimento_n400.csv (0.06 MB)


**Resumen ETL:** 88,682 → **88,008** limpios · lab n=400 · train=280 · test=120 · eval LLM=400